In [0]:
CREATE OR REPLACE TABLE cfpb_risk.gold.risk_alerts (
  alert_id STRING,
  institution_display_name STRING,
  rssd_id STRING,
  complaint_month DATE,
  product STRING,
  issue STRING,
  complaint_count BIGINT,
  prior_month_complaint_count BIGINT,
  rolling_3_month_avg DOUBLE,
  bank_total_complaints_monthly BIGINT,
  share_of_bank_complaints DOUBLE,
  severity_rating BIGINT,
  volume_points BIGINT,
  spike_points BIGINT,
  severity_points BIGINT,
  concentration_points BIGINT,
  risk_score BIGINT,
  alert_level STRING,
  alert_reason STRING,
  generated_ts TIMESTAMP
)
USING DELTA;

INSERT INTO cfpb_risk.gold.risk_alerts (
  alert_id,
  institution_display_name,
  rssd_id,
  complaint_month,
  product,
  issue,
  complaint_count,
  prior_month_complaint_count,
  rolling_3_month_avg,
  bank_total_complaints_monthly,
  share_of_bank_complaints,
  severity_rating,
  volume_points,
  spike_points,
  severity_points,
  concentration_points,
  risk_score,
  alert_level,
  alert_reason,
  generated_ts
)
WITH alerts_base AS (
  SELECT
    m.*,
    CAST(complaint_count AS DOUBLE) / NULLIF(rolling_3_month_avg, 0) AS spike_ratio,
    COALESCE(w.severity_rating, 1) AS severity_rating,
    COALESCE(w.severity_rating, 1) * 6 AS severity_points
  FROM cfpb_risk.gold.issue_clusters_monthly m
  LEFT JOIN cfpb_risk.reference.issue_weights w
    ON UPPER(TRIM(m.product)) = UPPER(TRIM(w.product))
   AND UPPER(TRIM(m.issue)) = UPPER(TRIM(w.issue))
),
alerts_points AS (
  SELECT
    sha2(concat_ws('|', rssd_id, product, issue, complaint_month), 256) AS alert_id,
    institution_display_name,
    rssd_id,
    complaint_month,
    product,
    issue,
    complaint_count,
    prior_month_complaint_count,
    rolling_3_month_avg,
    bank_total_complaints_monthly,
    share_of_bank_complaints,
    severity_rating,
    CASE
      WHEN complaint_count < 40 THEN 0
      WHEN complaint_count < 70 THEN 10
      WHEN complaint_count < 100 THEN 20
      ELSE 30
    END AS volume_points,
    CASE
      WHEN spike_ratio IS NULL THEN 0
      WHEN spike_ratio < 1.0 THEN 0
      WHEN spike_ratio < 1.1 THEN 10
      WHEN spike_ratio < 1.2 THEN 20
      ELSE 30
    END AS spike_points,
    severity_points,
    CASE 
      WHEN share_of_bank_complaints IS NULL THEN 0
      WHEN share_of_bank_complaints < 0.05 THEN 0
      WHEN share_of_bank_complaints < 0.08 THEN 10
      WHEN share_of_bank_complaints < 0.1 THEN 20
      ELSE 30
    END AS concentration_points,
    current_timestamp() AS generated_ts
  FROM alerts_base
),
alerts_scored AS (
  SELECT
    alert_id,
    institution_display_name,
    rssd_id,
    complaint_month,
    product,
    issue,
    complaint_count,
    prior_month_complaint_count,
    rolling_3_month_avg,
    bank_total_complaints_monthly,
    share_of_bank_complaints,
    severity_rating,
    volume_points,
    spike_points,
    severity_points,
    concentration_points,
    volume_points + spike_points + severity_points + concentration_points AS risk_score,
    generated_ts
  FROM alerts_points
  )
SELECT
  alert_id,
  institution_display_name,
  rssd_id,
  complaint_month,
  product,
  issue,
  complaint_count,
  prior_month_complaint_count,
  rolling_3_month_avg,
  bank_total_complaints_monthly,
  share_of_bank_complaints,
  severity_rating,
  volume_points,
  spike_points,
  severity_points,
  concentration_points,
  risk_score,
  CASE
    WHEN risk_score IS NULL THEN 'N/A'
    WHEN risk_score < 50 THEN 'Low'
    WHEN risk_score < 75 THEN 'Medium'
    WHEN risk_score < 110 THEN 'High'
    ELSE 'Urgent'
  END AS alert_level,
  CASE
    WHEN risk_score < 30 THEN 'Low risk detected'
    WHEN volume_points = GREATEST(volume_points, spike_points, severity_points, concentration_points) AND risk_score >= 30 THEN 'High complaint volume'
    WHEN spike_points = GREATEST(volume_points, spike_points, severity_points, concentration_points) AND risk_score >= 30 THEN 'Trend-driven increase'
    WHEN severity_points = GREATEST(volume_points, spike_points, severity_points, concentration_points)  AND risk_score >= 30 THEN 'High-severity product/issue category'
    WHEN concentration_points = GREATEST(volume_points, spike_points, severity_points, concentration_points)  AND risk_score >= 30 THEN 'Large share of product/issue within institution complaints'
    ELSE 'Multiple elevated risk factors'
  END AS alert_reason,
  generated_ts
FROM alerts_scored